# Act 1 — Cloud Build and Cloud Deploy

GCP's CI/CD story is two services: **Cloud Build** for build and test pipelines, **Cloud Deploy** for delivery pipelines. Most teams use both, with Cloud Build feeding artifacts into Cloud Deploy. Plenty of teams also use GitHub Actions / GitLab CI for the build stage and only Cloud Deploy for the delivery stage — that's a fine pattern too.

The headline GCP-specific bit is **Workload Identity Federation** (notebook 02), which lets GitHub Actions deploy to GCP without ever holding a service account JSON key. That single feature changes how the CI/CD plumbing looks.

## Cloud Build — build pipelines

Covered briefly in notebook 04; recap here in CD context.

- **`cloudbuild.yaml`** — list of container steps run in sequence (or in parallel with `waitFor`).
- **Triggers** — kick off builds on GitHub/GitLab push, Cloud Source Repos commit, Pub/Sub message, manual API call.
- **Substitutions** — variables like `$PROJECT_ID`, `$SHORT_SHA`, plus user-defined `_VARS`.
- **Private pools** — run builds inside your VPC. Required if the build needs to reach private resources (Cloud SQL Auth Proxy, internal package registries).
- **Artifacts** — pushed to Artifact Registry (containers, language packages).

**Vulnerability scanning** is automatic when images land in Artifact Registry. Findings hook into Binary Authorization (notebook 11) to gate deploys.

**Build provenance** generates an in-toto attestation (SLSA-compatible) signed by Cloud Build. That attestation is the input to Binary Authorization attestors — "this image was actually built by *our* Cloud Build pipeline."

## Cloud Deploy — delivery pipelines

**Cloud Deploy** is the delivery-pipeline product. You define a **delivery pipeline** as a YAML file: ordered **targets** (dev → staging → prod), promotion rules, approval gates, canary configuration. A **release** moves through the pipeline from target to target.

**Supports:**

- **Cloud Run services and jobs**
- **GKE** (Kubernetes manifests, Helm, kustomize)
- **Cloud Run for Anthos / GKE Autopilot**

**Canary patterns** are first-class:

- **Linear** — push to N% of traffic, wait, push to M%, etc.
- **Standard** — a single fixed canary step (e.g. 10% canary, then full).
- **Custom** — define your own phases.

**Verification phases** can run a Cloud Build job between canary phases — synthetic tests, smoke checks, SLO probes. Promotion is blocked until verification passes. Manual approval gates are the more cautious option.

**Why Cloud Deploy vs raw `gcloud` in Cloud Build:** Cloud Deploy keeps state across promotions, knows how to roll back, integrates with verification, and gives you an audit-friendly pipeline UI. For one-off deploys, `gcloud` is fine; for production pipelines with multiple environments, Cloud Deploy earns its keep.

## Workload Identity Federation for GitHub Actions

The canonical pattern: GitHub Actions deploys to GCP without holding a service account JSON key.

**Setup (one-time):**

1. Create a Workload Identity Pool (`acme-cicd-pool`).
2. Create a Provider in the pool that trusts GitHub's OIDC issuer. Use an `attribute_condition` that matches your repo and (ideally) environment: `attribute.repository == "acme/web" && attribute.environment == "prod"`.
3. Create a Google service account (`deployer-prod@…`). Grant it the runtime roles it needs (Cloud Run developer, Artifact Registry writer).
4. Grant `roles/iam.workloadIdentityUser` on `deployer-prod` to the WIF principal: `principalSet://iam.googleapis.com/.../attribute.repository/acme/web`.

**In the GitHub workflow:**

```yaml
- uses: google-github-actions/auth@v2
  with:
    workload_identity_provider: projects/.../providers/github-prod
    service_account: deployer-prod@acme.iam.gserviceaccount.com
- run: gcloud run deploy web --image=…
```

Underneath: GitHub OIDC token → STS exchange → `generateAccessToken` → SA access token → `gcloud` calls. Zero secrets in GitHub. Notebook 02 covered the dance in detail.

# Act 2 — Infrastructure as Code

ClickOps doesn't scale. You need infrastructure-as-code to track changes, review them, and roll them back. GCP works with three patterns: **Terraform** (the de-facto standard), **Config Connector** (Kubernetes-native), and the deprecating **Deployment Manager**.

## Terraform on GCP and Config Connector

**Terraform** with the `google` provider is the default infrastructure-as-code tool on GCP. Patterns worth knowing:

- **State storage** in GCS with state locking. Use a `terraform-state` project per environment.
- **Service account impersonation** via the `impersonate_service_account` provider field — Terraform runs as a deployer SA without a long-lived key. With WIF (above) this composes into key-less CI Terraform.
- **Workspaces** for environment isolation (dev/staging/prod). Or per-env directories — both work.
- **Modules** for reusable patterns (a `gke-cluster` module, a `cloud-run-service` module).

**Config Connector (KCC)** is the Kubernetes operator that exposes GCP resources as Kubernetes CRDs. You write a `ComputeInstance` YAML, apply it with `kubectl`, and Config Connector creates the actual VM. Used by teams who want one tool (Kubernetes) for both workloads and infrastructure.

**Deployment Manager** is GCP's original IaC product. Don't pick it for new work — Terraform has more ecosystem and Google's own docs increasingly default to Terraform examples.

# Act 3 — High availability and disaster recovery

Availability and DR aren't products; they're patterns you build by *configuring* the products from earlier notebooks. This act ties together the resilience story across compute, data, and network.

## HA patterns by component

| Component | Zonal | Regional | Multi-region |
|---|---|---|---|
| **Compute Engine MIG** | One zone | Three zones in region | Multiple regions + LB across them |
| **GKE** | Zonal cluster | Regional cluster (control plane across 3 zones) | Multiple clusters + multi-cluster ingress |
| **Cloud Run** | n/a — always regional | Default | Multiple regions + LB |
| **Cloud SQL** | Single zone | HA: primary + standby in 2 zones | Cross-region read replicas (manual promotion) |
| **AlloyDB** | n/a | Multi-zone in region by default | Cross-region replicas |
| **Spanner** | n/a | 3 zones | 3+ regions, synchronous |
| **Cloud Storage** | n/a — bucket location | Regional bucket | Dual-region or multi-region bucket |
| **BigQuery** | n/a | Single region | Multi-region |

**Most production workloads stop at regional.** Multi-region is for very high availability requirements (financial services, life safety) or genuinely global services. The cost and complexity step up significantly.

## DR strategies — RTO vs RPO

The Recovery Time Objective (RTO) is how long you can be down. The Recovery Point Objective (RPO) is how much data you can afford to lose. Four standard strategies trade RTO/RPO against cost:

| Strategy | RTO | RPO | Cost |
|---|---|---|---|
| **Backup & Restore** | Hours – days | Hours (last backup) | Lowest |
| **Pilot Light** | Tens of minutes | Minutes (replicated DB) | Low |
| **Warm Standby** | Minutes | Seconds | Medium |
| **Multi-site Active-Active** | Zero (failover automatic) | Zero | High |

**Map to GCP services:**

- **Backup & Restore** — GCS Archive class for cold backups, Cloud SQL automated backups (PITR), Backup and DR Service for VMs.
- **Pilot Light** — VPC + minimal infra pre-provisioned in DR region; Cloud SQL cross-region read replica; Cloud Run with min=0 in DR region waiting to scale up.
- **Warm Standby** — Same as Pilot Light but with non-trivial capacity already running and synced.
- **Active-Active** — Spanner multi-region, multi-region Cloud Storage, Cloud Run in multiple regions behind a Global External Application LB.

**Test the DR plan.** A plan you've never executed is not a plan. GameDay-style failovers monthly are the standard.

## Backup and DR Service

**Backup and DR Service** (formerly Actifio) is the managed backup service for VMs, databases, and file systems. It supports:

- **Compute Engine VMs** — application-consistent snapshots.
- **Cloud SQL** — beyond the built-in automated backups.
- **VMware Engine** — backups for the VMware-on-GCP workload.
- **File systems and databases** running on VMs (Oracle, SAP, MS SQL).

Used where the built-in per-service backup features aren't enough — typically enterprises with regulated backup retention policies.

# Act 4 — Migration

Four migration patterns worth naming. Most enterprises pick more than one in parallel — different workloads call for different approaches.

## Migration services

- **Database Migration Service (DMS)** — homogeneous and heterogeneous migrations between MySQL, PostgreSQL, SQL Server (on-prem or other clouds) and Cloud SQL / AlloyDB. Continuous replication during cutover, change-data-capture-based.
- **Migrate to Virtual Machines** — lift-and-shift VMware/Hyper-V VMs onto Compute Engine. Block-level replication; cutover with minimal downtime.
- **Migrate to Containers** — convert running VMs into containers that run on GKE. Used to escape VMs without rewriting apps. Niche but powerful.
- **Transfer Service / Transfer Appliance** — GCS data migration (notebook 05).
- **BigQuery Data Transfer Service** — recurring data transfers from SaaS sources (Google Ads, Salesforce, Teradata, Redshift) into BigQuery.

**The 7 Rs of migration** apply on GCP just like elsewhere: Rehost (lift-and-shift), Replatform (Cloud SQL instead of self-managed Postgres), Refactor (move to Cloud Run), Repurchase (replace with SaaS), Retire, Retain (leave it), Relocate. Most workloads land in Replatform or Refactor on GCP.

## What carries into later chapters

Notebook 14 wraps the course with the Architecture Framework, cost optimization, and ACE exam-prep notes. The patterns from this chapter — WIF for keyless CI, IaC, regional-first HA, DR strategy mapped to RTO/RPO — show up directly in the framework's Reliability and Operational Excellence pillars.

Three habits to carry forward:

- **Keyless CI via Workload Identity Federation.** Don't even create SA JSON keys for CI runners.
- **Regional everything in production.** Promotions to multi-region are intentional, not default.
- **DR is a tested plan, not a configuration.** Run the failover before you need to run it for real.